In [5]:
!pip install -q kagglehub

In [6]:
import kagglehub

path = kagglehub.dataset_download(
    "uciml/default-of-credit-card-clients-dataset"
)

print(path)

Using Colab cache for faster access to the 'default-of-credit-card-clients-dataset' dataset.
/kaggle/input/default-of-credit-card-clients-dataset


In [7]:
import os

print(os.listdir(path))

['UCI_Credit_Card.csv']


In [8]:
import pandas as pd

df = pd.read_csv(
    os.path.join(path, "UCI_Credit_Card.csv")
)

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [9]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 25)

Columns:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   300

In [11]:
df.isnull().sum().sum()

np.int64(0)

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
print("Target Distribution:")
print(df["default.payment.next.month"].value_counts())

print("\nTarget Percentage:")
print(df["default.payment.next.month"].value_counts(normalize=True) * 100)

Target Distribution:
default.payment.next.month
0    23364
1     6636
Name: count, dtype: int64

Target Percentage:
default.payment.next.month
0    77.88
1    22.12
Name: proportion, dtype: float64


In [14]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["ID", "default.payment.next.month"])
y = df["default.payment.next.month"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

X_train: (24000, 23)
X_test : (6000, 23)

Train target:
default.payment.next.month
0    18691
1     5309
Name: count, dtype: int64

Test target:
default.payment.next.month
0    4673
1    1327
Name: count, dtype: int64


In [15]:
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
models = {
    "Logistic Regression": (
        LogisticRegression(),
        X_train_scaled,
        X_test_scaled
    ),

    "SVM": (
        SVC(),
        X_train_scaled,
        X_test_scaled
    ),

    "KNN": (
        KNeighborsClassifier(),
        X_train_scaled,
        X_test_scaled
    ),

    "Random Forest": (
        RandomForestClassifier(),
        X_train,
        X_test
    ),

    "XGBoost": (
        XGBClassifier(),
        X_train,
        X_test
    )
}

In [18]:
results = []

for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)

    y_pred = model.predict(Xte)

    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

    print(f"{name}: {accuracy:.4f}")

Logistic Regression: 0.8077
SVM: 0.8160
KNN: 0.7928
Random Forest: 0.8137
XGBoost: 0.8108


In [19]:
baseline_results = pd.DataFrame(results)

baseline_results = baseline_results.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

baseline_results

,Model,Accuracy
0,SVM,0.816000
1,Random Forest,0.813667
2,XGBoost,0.810833
3,Logistic Regression,0.807667
4,KNN,0.792833


In [20]:
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (X_train < lower_bound) |
    (X_train > upper_bound)
).any(axis=1)

print("Outliers:", outlier_mask.sum())
print("Remaining:", (~outlier_mask).sum())

Outliers: 13714
Remaining: 10286


In [21]:
X_train_iqr = X_train.loc[~outlier_mask].copy()
y_train_iqr = y_train.loc[~outlier_mask].copy()

print("X_train:", X_train_iqr.shape)
print("y_train:", y_train_iqr.shape)

X_train: (10286, 23)
y_train: (10286,)


In [22]:
scaler_iqr = StandardScaler()

X_train_iqr_scaled = scaler_iqr.fit_transform(X_train_iqr)
X_test_iqr_scaled = scaler_iqr.transform(X_test)

In [23]:
svm_iqr = SVC()

svm_iqr.fit(X_train_iqr_scaled, y_train_iqr)

y_pred_iqr = svm_iqr.predict(X_test_iqr_scaled)

iqr_accuracy = accuracy_score(y_test, y_pred_iqr)

print(f"IQR SVM Accuracy: {iqr_accuracy:.4f}")

IQR SVM Accuracy: 0.7788


In [25]:
import numpy as np
from scipy.stats import zscore

z_scores = np.abs(zscore(X_train))

outlier_mask_z = (z_scores > 3).any(axis=1)

print("Outliers:", outlier_mask_z.sum())
print("Remaining:", (~outlier_mask_z).sum())

Outliers: 2910
Remaining: 21090


In [26]:
X_train_z = X_train.loc[~outlier_mask_z].copy()
y_train_z = y_train.loc[~outlier_mask_z].copy()

print("X_train:", X_train_z.shape)
print("y_train:", y_train_z.shape)

X_train: (21090, 23)
y_train: (21090,)


In [27]:
scaler_z = StandardScaler()

X_train_z_scaled = scaler_z.fit_transform(X_train_z)
X_test_z_scaled = scaler_z.transform(X_test)

In [28]:
svm_z = SVC()

svm_z.fit(X_train_z_scaled, y_train_z)

y_pred_z = svm_z.predict(X_test_z_scaled)

z_accuracy = accuracy_score(y_test, y_pred_z)

print(f"Z-Score SVM Accuracy: {z_accuracy:.4f}")

Z-Score SVM Accuracy: 0.8130


In [29]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination="auto",
    random_state=42
)

outlier_pred = iso.fit_predict(X_train)

outlier_mask_iso = outlier_pred == -1

print("Outliers:", outlier_mask_iso.sum())
print("Remaining:", (~outlier_mask_iso).sum())

Outliers: 1489
Remaining: 22511


In [30]:
X_train_iso = X_train.loc[~outlier_mask_iso].copy()
y_train_iso = y_train.loc[~outlier_mask_iso].copy()

print("X_train:", X_train_iso.shape)
print("y_train:", y_train_iso.shape)

X_train: (22511, 23)
y_train: (22511,)


In [31]:
scaler_iso = StandardScaler()

X_train_iso_scaled = scaler_iso.fit_transform(X_train_iso)
X_test_iso_scaled = scaler_iso.transform(X_test)

In [32]:
svm_iso = SVC()

svm_iso.fit(X_train_iso_scaled, y_train_iso)

y_pred_iso = svm_iso.predict(X_test_iso_scaled)

iso_accuracy = accuracy_score(y_test, y_pred_iso)

print(f"Isolation Forest SVM Accuracy: {iso_accuracy:.4f}")

Isolation Forest SVM Accuracy: 0.8130


In [33]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor()

outlier_mask_lof = lof.fit_predict(X_train) == -1

print("Outliers:", outlier_mask_lof.sum())
print("Remaining:", (~outlier_mask_lof).sum())

Outliers: 5580
Remaining: 18420


In [34]:
X_train_lof = X_train.loc[~outlier_mask_lof].copy()
y_train_lof = y_train.loc[~outlier_mask_lof].copy()

print("X_train:", X_train_lof.shape)
print("y_train:", y_train_lof.shape)

X_train: (18420, 23)
y_train: (18420,)


In [35]:
scaler_lof = StandardScaler()

X_train_lof_scaled = scaler_lof.fit_transform(X_train_lof)
X_test_lof_scaled = scaler_lof.transform(X_test)

In [36]:
svm_lof = SVC()

svm_lof.fit(X_train_lof_scaled, y_train_lof)

y_pred_lof = svm_lof.predict(X_test_lof_scaled)
lof_accuracy = accuracy_score(y_test, y_pred_lof)

print(f"LOF SVM Accuracy: {lof_accuracy:.4f}")

LOF SVM Accuracy: 0.8163


In [37]:
from sklearn.covariance import EllipticEnvelope

ee = EllipticEnvelope(random_state=42)

outlier_mask_ee = ee.fit_predict(X_train) == -1

print("Outliers:", outlier_mask_ee.sum())
print("Remaining:", (~outlier_mask_ee).sum())

/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (207.696388106379516 > 142.797645693144347). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (208.581187909027392 > 145.997886596950877). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (207.978452875060896 > 145.316277675052788). You may want to try with a higher value of support_fraction (current value: 0.501).
  warnings.warn(
/usr/local/lib/p

Outliers: 2400
Remaining: 21600


In [38]:
X_train_ee = X_train.loc[~outlier_mask_ee].copy()
y_train_ee = y_train.loc[~outlier_mask_ee].copy()

scaler_ee = StandardScaler()

X_train_ee_scaled = scaler_ee.fit_transform(X_train_ee)
X_test_ee_scaled = scaler_ee.transform(X_test)

In [39]:
svm_ee = SVC()

svm_ee.fit(X_train_ee_scaled, y_train_ee)

y_pred_ee = svm_ee.predict(X_test_ee_scaled)

ee_accuracy = accuracy_score(y_test, y_pred_ee)

print(f"Elliptic Envelope SVM Accuracy: {ee_accuracy:.4f}")

Elliptic Envelope SVM Accuracy: 0.8143


In [40]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

ros = RandomOverSampler(random_state=42)

X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_ros.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {0: 18691, 1: 18691}


In [41]:
scaler_ros = StandardScaler()

X_train_ros_scaled = scaler_ros.fit_transform(X_train_ros)
X_test_ros_scaled = scaler_ros.transform(X_test)

In [42]:
svm_ros = SVC()

svm_ros.fit(X_train_ros_scaled, y_train_ros)

y_pred_ros = svm_ros.predict(X_test_ros_scaled)

ros_accuracy = accuracy_score(y_test, y_pred_ros)

print(f"Random Over Sampling SVM Accuracy: {ros_accuracy:.4f}")

Random Over Sampling SVM Accuracy: 0.7745


In [43]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_rus.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {0: 5309, 1: 5309}


In [44]:
scaler_rus = StandardScaler()

X_train_rus_scaled = scaler_rus.fit_transform(X_train_rus)
X_test_rus_scaled = scaler_rus.transform(X_test)

In [45]:
svm_rus = SVC()

svm_rus.fit(X_train_rus_scaled, y_train_rus)

y_pred_rus = svm_rus.predict(X_test_rus_scaled)

rus_accuracy = accuracy_score(y_test, y_pred_rus)

print(f"Random Under Sampling SVM Accuracy: {rus_accuracy:.4f}")

Random Under Sampling SVM Accuracy: 0.7750


In [46]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_smote.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {0: 18691, 1: 18691}


In [47]:
scaler_smote = StandardScaler()

X_train_smote_scaled = scaler_smote.fit_transform(X_train_smote)
X_test_smote_scaled = scaler_smote.transform(X_test)

In [48]:
svm_smote = SVC()

svm_smote.fit(X_train_smote_scaled, y_train_smote)

y_pred_smote = svm_smote.predict(X_test_smote_scaled)

smote_accuracy = accuracy_score(y_test, y_pred_smote)

print(f"SMOTE SVM Accuracy: {smote_accuracy:.4f}")

SMOTE SVM Accuracy: 0.7422


In [49]:
from imblearn.over_sampling import ADASYN

adasyn = ADASYN(random_state=42)

X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_adasyn.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {1: 19252, 0: 18691}


In [50]:
scaler_adasyn = StandardScaler()

X_train_adasyn_scaled = scaler_adasyn.fit_transform(X_train_adasyn)
X_test_adasyn_scaled = scaler_adasyn.transform(X_test)

In [51]:
svm_adasyn = SVC()

svm_adasyn.fit(X_train_adasyn_scaled, y_train_adasyn)

y_pred_adasyn = svm_adasyn.predict(X_test_adasyn_scaled)

adasyn_accuracy = accuracy_score(y_test, y_pred_adasyn)

print(f"ADASYN SVM Accuracy: {adasyn_accuracy:.4f}")

ADASYN SVM Accuracy: 0.7253


In [52]:
from imblearn.combine import SMOTEENN

smoteenn = SMOTEENN(random_state=42)

X_train_smoteenn, y_train_smoteenn = smoteenn.fit_resample(
    X_train,
    y_train
)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_smoteenn.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {1: 13706, 0: 8219}


In [53]:
scaler_smoteenn = StandardScaler()

X_train_smoteenn_scaled = scaler_smoteenn.fit_transform(X_train_smoteenn)
X_test_smoteenn_scaled = scaler_smoteenn.transform(X_test)

In [54]:
svm_smoteenn = SVC()

svm_smoteenn.fit(X_train_smoteenn_scaled, y_train_smoteenn)

y_pred_smoteenn = svm_smoteenn.predict(X_test_smoteenn_scaled)

smoteenn_accuracy = accuracy_score(y_test, y_pred_smoteenn)

print(f"SMOTEENN SVM Accuracy: {smoteenn_accuracy:.4f}")

SMOTEENN SVM Accuracy: 0.6282


In [55]:
from imblearn.over_sampling import BorderlineSMOTE

borderline_smote = BorderlineSMOTE(random_state=42)

X_train_bsmote, y_train_bsmote = borderline_smote.fit_resample(
    X_train,
    y_train
)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_bsmote.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {0: 18691, 1: 18691}


In [56]:
scaler_bsmote = StandardScaler()

X_train_bsmote_scaled = scaler_bsmote.fit_transform(X_train_bsmote)
X_test_bsmote_scaled = scaler_bsmote.transform(X_test)

In [57]:
svm_bsmote = SVC()

svm_bsmote.fit(X_train_bsmote_scaled, y_train_bsmote)

y_pred_bsmote = svm_bsmote.predict(X_test_bsmote_scaled)

bsmote_accuracy = accuracy_score(y_test, y_pred_bsmote)

print(f"Borderline-SMOTE SVM Accuracy: {bsmote_accuracy:.4f}")

Borderline-SMOTE SVM Accuracy: 0.7380


In [58]:
from imblearn.under_sampling import TomekLinks

tomek = TomekLinks()

X_train_tomek, y_train_tomek = tomek.fit_resample(
    X_train,
    y_train
)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_tomek.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {0: 17248, 1: 5309}


In [59]:
scaler_tomek = StandardScaler()

X_train_tomek_scaled = scaler_tomek.fit_transform(X_train_tomek)
X_test_tomek_scaled = scaler_tomek.transform(X_test)

In [60]:
svm_tomek = SVC()

svm_tomek.fit(X_train_tomek_scaled, y_train_tomek)

y_pred_tomek = svm_tomek.predict(X_test_tomek_scaled)

tomek_accuracy = accuracy_score(y_test, y_pred_tomek)

print(f"Tomek Links SVM Accuracy: {tomek_accuracy:.4f}")

Tomek Links SVM Accuracy: 0.8155


In [61]:
from imblearn.combine import SMOTETomek

smotetomek = SMOTETomek(random_state=42)

X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(
    X_train,
    y_train
)

print("Before:", y_train.value_counts().to_dict())
print("After :", y_train_smotetomek.value_counts().to_dict())

Before: {0: 18691, 1: 5309}
After : {0: 18179, 1: 18179}


In [62]:
scaler_smotetomek = StandardScaler()

X_train_smotetomek_scaled = scaler_smotetomek.fit_transform(
    X_train_smotetomek
)

X_test_smotetomek_scaled = scaler_smotetomek.transform(X_test)

In [63]:
svm_smotetomek = SVC()

svm_smotetomek.fit(
    X_train_smotetomek_scaled,
    y_train_smotetomek
)

y_pred_smotetomek = svm_smotetomek.predict(
    X_test_smotetomek_scaled
)

smotetomek_accuracy = accuracy_score(
    y_test,
    y_pred_smotetomek
)

print(f"SMOTETomek SVM Accuracy: {smotetomek_accuracy:.4f}")

SMOTETomek SVM Accuracy: 0.7408


In [64]:
scaler_cw = StandardScaler()

X_train_cw_scaled = scaler_cw.fit_transform(X_train)
X_test_cw_scaled = scaler_cw.transform(X_test)

In [65]:
svm_cw = SVC(class_weight="balanced")

svm_cw.fit(X_train_cw_scaled, y_train)

y_pred_cw = svm_cw.predict(X_test_cw_scaled)

cw_accuracy = accuracy_score(y_test, y_pred_cw)

print(f"Class Weight SVM Accuracy: {cw_accuracy:.4f}")

Class Weight SVM Accuracy: 0.7763


In [66]:
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
import numpy as np

components_list = [2,3,5, 10, 15, 20]

pca_results = []

for n in components_list:
    pca = PCA(n_components=n)

    X_train_pca = pca.fit_transform(X_train_scaled)

    svm = SVC()

    scores = cross_val_score(
        svm,
        X_train_pca,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    pca_results.append({
        "Components": n,
        "CV Accuracy": scores.mean()
    })

pca_results

[{'Components': 2, 'CV Accuracy': np.float64(0.8034583333333334)},
 {'Components': 3, 'CV Accuracy': np.float64(0.8031666666666666)},
 {'Components': 5, 'CV Accuracy': np.float64(0.803375)},
 {'Components': 10, 'CV Accuracy': np.float64(0.8059166666666666)},
 {'Components': 15, 'CV Accuracy': np.float64(0.8164583333333333)},
 {'Components': 20, 'CV Accuracy': np.float64(0.81975)}]

In [67]:
pca_results_df = pd.DataFrame(pca_results)

print(pca_results_df)

best_n = pca_results_df.loc[
    pca_results_df["CV Accuracy"].idxmax(),
    "Components"
]

print("Best Components:", best_n)

   Components  CV Accuracy
0           2     0.803458
1           3     0.803167
2           5     0.803375
3          10     0.805917
4          15     0.816458
5          20     0.819750
Best Components: 20


In [68]:
pca_best = PCA(n_components=best_n)

X_train_pca = pca_best.fit_transform(X_train_scaled)
X_test_pca = pca_best.transform(X_test_scaled)

svm_pca = SVC()

svm_pca.fit(X_train_pca, y_train)

y_pred_pca = svm_pca.predict(X_test_pca)

pca_accuracy = accuracy_score(y_test, y_pred_pca)

print(f"PCA SVM Accuracy: {pca_accuracy:.4f}")

PCA SVM Accuracy: 0.8158


In [69]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=1)

X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

print("Original features:", X_train_scaled.shape[1])
print("LDA features:", X_train_lda.shape[1])

Original features: 23
LDA features: 1


In [70]:
svm_lda = SVC()

svm_lda.fit(X_train_lda, y_train)

y_pred_lda = svm_lda.predict(X_test_lda)

lda_accuracy = accuracy_score(y_test, y_pred_lda)

print(f"LDA SVM Accuracy: {lda_accuracy:.4f}")

LDA SVM Accuracy: 0.8122


In [77]:
from sklearn.neighbors import LocalOutlierFactor

lof_final = LocalOutlierFactor()

outlier_mask_final = lof_final.fit_predict(X_train) == -1

X_train_final = X_train.loc[~outlier_mask_final].copy()
y_train_final = y_train.loc[~outlier_mask_final].copy()

print("After LOF:", X_train_final.shape)
print("Target:", y_train_final.value_counts().to_dict())

After LOF: (18420, 23)
Target: {0: 14201, 1: 4219}


In [78]:
from imblearn.under_sampling import TomekLinks

tomek_final = TomekLinks()

X_train_final, y_train_final = tomek_final.fit_resample(
    X_train_final,
    y_train_final
)

print("After Tomek:", X_train_final.shape)
print("Target:", y_train_final.value_counts().to_dict())

After Tomek: (17261, 23)
Target: {0: 13042, 1: 4219}


In [79]:
scaler_final = StandardScaler()

X_train_final_scaled = scaler_final.fit_transform(X_train_final)
X_test_final_scaled = scaler_final.transform(X_test)

In [80]:
pca_final = PCA(n_components=20)

X_train_final_pca = pca_final.fit_transform(X_train_final_scaled)
X_test_final_pca = pca_final.transform(X_test_final_scaled)

print("Final train shape:", X_train_final_pca.shape)
print("Final test shape:", X_test_final_pca.shape)

Final train shape: (17261, 20)
Final test shape: (6000, 20)


In [81]:
svm_final = SVC()

svm_final.fit(X_train_final_pca, y_train_final)

y_pred_final = svm_final.predict(X_test_final_pca)

final_accuracy = accuracy_score(y_test, y_pred_final)

print(f"Final Combination Accuracy: {final_accuracy:.4f}")

Final Combination Accuracy: 0.8177


In [84]:
import pandas as pd
import numpy as np

train_corr = X_train.copy()
train_corr["target"] = y_train

correlations = train_corr.corr()["target"].drop("target").abs()
correlations = correlations.sort_values(ascending=False)

print(correlations)

PAY_0        0.327671
PAY_2        0.265527
PAY_3        0.238930
PAY_4        0.220063
PAY_5        0.203260
PAY_6        0.188717
LIMIT_BAL    0.160995
PAY_AMT1     0.073441
PAY_AMT2     0.065078
PAY_AMT4     0.055313
PAY_AMT3     0.053863
PAY_AMT6     0.053280
PAY_AMT5     0.052609
SEX          0.043862
EDUCATION    0.026127
BILL_AMT1    0.022720
MARRIAGE     0.019661
BILL_AMT3    0.018280
BILL_AMT2    0.018048
BILL_AMT4    0.013726
AGE          0.012830
BILL_AMT5    0.009710
BILL_AMT6    0.008177
Name: target, dtype: float64


In [86]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

thresholds = [0.01,0.05, 0.10, 0.15, 0.20]

correlation_results = []

for threshold in thresholds:

    selected_features = correlations[correlations >= threshold].index

    X_train_corr = X_train[selected_features]
    X_test_corr = X_test[selected_features]

    scaler_corr = StandardScaler()
    X_train_corr_scaled = scaler_corr.fit_transform(X_train_corr)
    X_test_corr_scaled = scaler_corr.transform(X_test_corr)

    svm_corr = SVC()
    svm_corr.fit(X_train_corr_scaled, y_train)

    y_pred_corr = svm_corr.predict(X_test_corr_scaled)

    accuracy = accuracy_score(y_test, y_pred_corr)

    correlation_results.append({
        "Threshold": threshold,
        "Features": len(selected_features),
        "Accuracy": accuracy
    })

    print(
        f"Threshold: {threshold:.2f} | "
        f"Features: {len(selected_features)} | "
        f"Accuracy: {accuracy:.4f}"
    )

Threshold: 0.01 | Features: 21 | Accuracy: 0.8162
Threshold: 0.05 | Features: 13 | Accuracy: 0.8177
Threshold: 0.10 | Features: 7 | Accuracy: 0.8172
Threshold: 0.15 | Features: 7 | Accuracy: 0.8172
Threshold: 0.20 | Features: 5 | Accuracy: 0.8168


In [93]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_selected_scaled,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [94]:
C_values = [0.1, 1, 10]
gamma_values = ["scale", 0.01, 0.1]

results = []

for C in C_values:
    for gamma in gamma_values:

        svm = SVC(
            C=C,
            gamma=gamma,
            kernel="rbf"
        )

        svm.fit(X_train_sub, y_train_sub)

        y_val_pred = svm.predict(X_val)
        accuracy = accuracy_score(y_val, y_val_pred)

        results.append({
            "C": C,
            "Gamma": gamma,
            "Validation Accuracy": accuracy
        })

        print(
            f"C={C:<4} | "
            f"Gamma={str(gamma):<6} | "
            f"Validation Accuracy={accuracy:.4f}"
        )

C=0.1  | Gamma=scale  | Validation Accuracy=0.8175
C=0.1  | Gamma=0.01   | Validation Accuracy=0.8077
C=0.1  | Gamma=0.1    | Validation Accuracy=0.8175
C=1    | Gamma=scale  | Validation Accuracy=0.8194
C=1    | Gamma=0.01   | Validation Accuracy=0.8154
C=1    | Gamma=0.1    | Validation Accuracy=0.8185
C=10   | Gamma=scale  | Validation Accuracy=0.8163
C=10   | Gamma=0.01   | Validation Accuracy=0.8146
C=10   | Gamma=0.1    | Validation Accuracy=0.8156


In [95]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

best_svm = SVC(
    C=1,
    gamma="scale",
    kernel="rbf"
)

best_svm.fit(X_train_selected_scaled, y_train)

y_pred_svm = best_svm.predict(X_test_selected_scaled)

svm_test_accuracy = accuracy_score(y_test, y_pred_svm)

print(f"Final Tuned SVM Test Accuracy: {svm_test_accuracy:.4f}")

Final Tuned SVM Test Accuracy: 0.8177


In [96]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_selected,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [97]:
n_estimators_values = [100, 200, 300]
max_depth_values = [None, 10, 20]

rf_results = []

for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:

        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_train_sub, y_train_sub)

        y_val_pred = rf.predict(X_val)
        accuracy = accuracy_score(y_val, y_val_pred)

        rf_results.append({
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "Validation Accuracy": accuracy
        })

        print(
            f"Trees={n_estimators:<3} | "
            f"Max Depth={str(max_depth):<4} | "
            f"Validation Accuracy={accuracy:.4f}"
        )

Trees=100 | Max Depth=None | Validation Accuracy=0.8121
Trees=100 | Max Depth=10   | Validation Accuracy=0.8158
Trees=100 | Max Depth=20   | Validation Accuracy=0.8177
Trees=200 | Max Depth=None | Validation Accuracy=0.8119
Trees=200 | Max Depth=10   | Validation Accuracy=0.8165
Trees=200 | Max Depth=20   | Validation Accuracy=0.8163
Trees=300 | Max Depth=None | Validation Accuracy=0.8131
Trees=300 | Max Depth=10   | Validation Accuracy=0.8183
Trees=300 | Max Depth=20   | Validation Accuracy=0.8148


In [98]:
best_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_selected, y_train)

y_pred_rf = best_rf.predict(X_test_selected)

rf_test_accuracy = accuracy_score(y_test, y_pred_rf)

print(f"Final Tuned Random Forest Accuracy: {rf_test_accuracy:.4f}")

Final Tuned Random Forest Accuracy: 0.8183
